# Notebook 2 — Classification LOS (Line of Sight)
**Projet FerroMobile / Flexy** — UTBM FEMTO-ST DISC/OMNI  
Ligne ferroviaire 785000 · Courpière–Ambert · 30 km

**Objectif** : classifier chaque point de la ligne selon la qualité de visibilité radio entre le train et les antennes cellulaires.

**Label** : `los_category` ∈ {0 = Tunnel, 1 = NLOS bloqué, 2 = Partiel/dégradé, 3 = LOS dégagé}  
**Features** : altitude, végétation, distance antenne, météo  
**Note** : `los_category` est calculé dans le pipeline depuis le profil de terrain réel (EU-DEM) — c'est une variable terrain, pas une métrique radio simulée.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score)
from sklearn.preprocessing import label_binarize
import shap

print("Librairies chargées.")


## 1. Chargement et agrégation

In [ ]:
df = pd.read_csv('dataset_enrichi.csv', low_memory=False)
print(f"Dataset brut : {df.shape}")

# Agrégation par point — on prend le LOS minimum (pire cas parmi toutes les antennes)
by_point = df.groupby('point_id').agg(
    lat          = ('lat',          'first'),
    lon          = ('lon',          'first'),
    distance_km  = ('distance_km',  'first'),
    altitude_m   = ('altitude_m',   'first'),
    veg_num      = ('veg_num',      'first'),
    vegetation   = ('vegetation',   'first'),
    in_tunnel    = ('in_tunnel',    'max'),
    los_category = ('los_category', 'min'),   # pire cas LOS
    dist_ant_km  = ('dist_ant_km',  'min'),
    n_antennas   = ('ant_id',       'count'),
    pluie_norm   = ('pluie_norm',   'mean'),
    temp_norm    = ('temp_norm',    'mean'),
).reset_index()

LOS_LABELS = {0: 'Tunnel', 1: 'NLOS bloqué', 2: 'Partiel/dégradé', 3: 'LOS dégagé'}
print(f"\nDistribution LOS par point :")
for cat, label in LOS_LABELS.items():
    n = (by_point['los_category'] == cat).sum()
    print(f"  {cat} — {label:20s} : {n:4d} points ({n/len(by_point)*100:.1f}%)")


## 2. Features et split spatial

In [ ]:
FEATURES = [
    'altitude_m',    # EU-DEM réel
    'veg_num',       # ESA WorldCover réel
    'dist_ant_km',   # distance à l'antenne la plus proche
    'n_antennas',    # nombre d'antennes disponibles
    'pluie_norm',    # précipitations normalisées ERA5
    'temp_norm',     # température normalisée ERA5
]

# Imputation NaN
for col in FEATURES:
    by_point[col] = by_point[col].fillna(by_point[col].median())

X = by_point[FEATURES].copy()
y = by_point['los_category'].copy()

# Split spatial 80/20 par distance_km
by_point_sorted = by_point.sort_values('distance_km').reset_index(drop=True)
split_idx = int(len(by_point_sorted) * 0.8)

X_train = by_point_sorted.loc[:split_idx-1, FEATURES]
X_test  = by_point_sorted.loc[split_idx:,   FEATURES]
y_train = by_point_sorted.loc[:split_idx-1, 'los_category']
y_test  = by_point_sorted.loc[split_idx:,   'los_category']

print(f"Train : {len(X_train)} points")
print(f"Test  : {len(X_test)} points")
print(f"\nDistribution train :")
print(y_train.value_counts().sort_index().rename(LOS_LABELS))
print(f"\nDistribution test :")
print(y_test.value_counts().sort_index().rename(LOS_LABELS))


## 3. Entraînement — Random Forest multi-classe

In [ ]:
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                             min_samples_leaf=2, random_state=42)
rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)

print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred,
      target_names=list(LOS_LABELS.values())))

# AUC macro (one-vs-rest)
y_test_bin = label_binarize(y_test, classes=[0,1,2,3])
auc_macro  = roc_auc_score(y_test_bin, y_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC macro (OvR) : {auc_macro:.4f}")


## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
cm   = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])
disp = ConfusionMatrixDisplay(cm, display_labels=list(LOS_LABELS.values()))
disp.plot(ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=30)
axes[0].set_title('Matrice de Confusion — LOS multi-classe', fontweight='bold')

# Importance des features
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
axes[1].barh(fi.index, fi.values, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Importance')
axes[1].set_title('Importance des features (Random Forest)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Notebook 2 — Classification LOS', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('nb2_resultats.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : nb2_resultats.png")


## 5. Carte LOS sur la ligne

In [ ]:
X_all = by_point[FEATURES].copy()
for col in FEATURES:
    X_all[col] = X_all[col].fillna(X_all[col].median())

by_point['pred_los'] = rf.predict(X_all)

COLORS_LOS = {0: 'black', 1: 'crimson', 2: 'darkorange', 3: 'steelblue'}
LABELS_LOS  = {0: 'Tunnel', 1: 'NLOS bloqué', 2: 'Partiel/dégradé', 3: 'LOS dégagé'}

fig, ax = plt.subplots(figsize=(18, 4))
for cat, color in COLORS_LOS.items():
    mask = by_point['pred_los'] == cat
    ax.scatter(by_point.loc[mask, 'distance_km'],
               by_point.loc[mask, 'altitude_m'],
               c=color, s=12, alpha=0.85, label=LABELS_LOS[cat])

ax.set_xlabel('Distance (km)', fontsize=12)
ax.set_ylabel('Altitude (m)', fontsize=12)
ax.set_title('Classification LOS sur la ligne 785000 — Courpière–Ambert', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nb2_carte_los.png', dpi=150, bbox_inches='tight')
plt.show()
print("Carte sauvegardée : nb2_carte_los.png")
